In [1]:
import numpy as np
import math
import matplotlib.pyplot as plt
import time 
from scipy.stats import norm
import QuantLib as ql

In [ ]:
def gbm_path_sim(s_start, r, sigma, T_sim, n_sims, m_steps):
    """
    Simulates Geometric Brownian Motion paths using an iterative (for-loop) approach.
    
    s_start: Initial asset price (S_t)
    r: Risk-free interest rate
    sigma: Volatility
    T_sim: Time horizon for the simulation (e.g., T_remaining)
    n_sims: Number of simulation paths (N)
    m_steps: Number of time steps (M_remaining)
    """
    
    # Create the matrix to hold all paths
    # We need m_steps + 1 columns to hold the initial price S_t
    simulated_paths = np.zeros((n_sims, m_steps + 1))
    
    dt = T_sim / m_steps
    
    # Pre-calculate the fixed drift and diffusion components
    drift = (r - 0.5 * sigma**2) * dt
    diffusion = sigma * math.sqrt(dt)
    
    # Outer loop for each simulation path
    for i in range(n_sims):
        
        # Generate all random numbers for this path at once
        norm_obs = np.random.normal(0.0, 1.0, m_steps)
        
        # Set the starting price for this path
        simulated_paths[i, 0] = s_start
        
        # Inner loop for each time step
        for j in range(m_steps):
            simulated_paths[i, j + 1] = simulated_paths[i, j] * np.exp(drift + diffusion * norm_obs[j])
            
    # Return the complete matrix of paths
    return simulated_paths

def gbm_path_sim_vectorized(
    s_start, r, sigma, T_sim, n_sims, m_steps, Z_matrix=None
):
    """
    Simulates Geometric Brownian Motion paths using a vectorized numpy approach.
    
    Can optionally accept a pre-generated matrix of standard normal
    random numbers (Z_matrix) for variance reduction techniques.
    
    s_start: Initial asset price (S_t)
    r: Risk-free interest rate
    sigma: Volatility
    T_sim: Time horizon for the simulation (e.g., T_remaining)
    n_sims: Number of simulation paths (N)
    m_steps: Number of time steps (M_remaining)
    Z_matrix (np.ndarray, optional): A pre-generated (n_sims, m_steps) matrix
                                     of standard normal random numbers.
    """
    
    dt = T_sim / m_steps
    
    # 1. Get the random numbers
    if Z_matrix is None:
        # If no matrix is provided, generate one
        Z = np.random.normal(0.0, 1.0, (n_sims, m_steps))
    else:
        # If a matrix is provided, use it
        # (A good practice check to ensure shapes match)
        if Z_matrix.shape != (n_sims, m_steps):
            raise ValueError(f"Z_matrix shape {Z_matrix.shape} does not match (n_sims, m_steps) {(n_sims, m_steps)}")
        Z = Z_matrix
    
    # 2. Calculate all log-returns in one go
    # Shape: (n_sims, m_steps)
    log_returns = (r - 0.5 * sigma**2) * dt + sigma * math.sqrt(dt) * Z
    
    # 3. Calculate the cumulative sum of log-returns across time (axis=1)
    # Shape: (n_sims, m_steps)
    cumulative_log_returns = np.cumsum(log_returns, axis=1)
    
    # 4. Create the output matrix and set the start price
    # Shape: (n_sims, m_steps + 1)
    simulated_paths = np.zeros((n_sims, m_steps + 1))
    simulated_paths[:, 0] = s_start
    
    # 5. Exponentiate and multiply by s_start to get all prices from t=1 to t=M
    # This fills all columns from index 1 onwards
    simulated_paths[:, 1:] = s_start * np.exp(cumulative_log_returns)
            
    # Return the complete matrix of paths
    return simulated_paths

def bsm_price(s_start, k, r, sigma, t_current, T_maturity, omega):
    """
    Calculates the Black-Scholes-Merton price for European options
    with consistent argument names.

    This function is vectorized. 's_start', 'k', 'r', and 'sigma' can be
    numpy arrays, while 't_current', 'T_maturity', and 'omega' are scalars.

    Args:
        s_start (float or np.ndarray): Current spot price(s) of the underlying.
        k (float or np.ndarray): Strike price(s).
        r (float or np.ndarray): Risk-free interest rate(s).
        sigma (float or np.ndarray): Volatility (annualized).
        t_current (float): Current time (in years).
        T_maturity (float): Time to maturity (in years).
        omega (int): Option type: 1 for a call, -1 for a put.

    Returns:
        float or np.ndarray: The BSM price(s) of the option(s).
    """
    
    # Input Validation
    if omega not in (1, -1):
        raise ValueError("omega must be 1 (call) or -1 (put)")

    tau = T_maturity - t_current
    
    if np.any(tau <= 0):
        raise ValueError("T_maturity must be greater than t_current (tau must be > 0)")

    # BSM Formula
    
    # Calculate d1 and d2
    d_1 = (np.log(s_start / k) + (r + 0.5 * sigma**2) * tau) / (sigma * np.sqrt(tau))
    d_2 = d_1 - sigma * np.sqrt(tau)

    # Calculate price using the generalized call/put formula
    # norm.cdf() is the Python equivalent of R's pnorm()
    price = omega * (s_start * norm.cdf(omega * d_1) - k * np.exp(-r * tau) * norm.cdf(omega * d_2))

    return price

def calculate_mtm_matrix_european(
    all_paths: np.ndarray, 
    k: float, 
    r: float, 
    sigma: float, 
    T_maturity: float, 
    omega: int
):
    """
    Calculates the MtM matrix for a European option given pre-simulated paths.
    
    Args:
        all_paths (np.ndarray): The (n_sims, m_steps + 1) matrix of simulated prices.
        k (float): Strike price.
        r (float): Risk-free rate.
        sigma (float): Volatility.
        T_maturity (float): Original time to maturity.
        omega (int): 1 for call, -1 for put.

    Returns:
        mtm_matrix (np.ndarray): The (n_sims, m_steps + 1) matrix of MtM values.
        time_vector (np.ndarray): The (m_steps + 1) vector of time points.
    """
    
    # 1. Get shape and time info from the paths matrix
    n_sims, m_steps_plus_1 = all_paths.shape
    m_steps = m_steps_plus_1 - 1
    
    # 2. Create helper variables
    mtm_matrix = np.zeros_like(all_paths)
    time_vector = np.linspace(0, T_maturity, m_steps + 1)
    
    print("Calculating MtM matrix (pricing at each time step)...")
    # 3. Loop through TIME (columns), not paths
    # We loop from t=0 up to t=M-1 (the step *before* maturity)
    for j in range(m_steps):
        t_current = time_vector[j]
        
        # Get the vector of all spot prices at this time step
        s_at_tj = all_paths[:, j]
        
        # Call the BSM pricer ONCE on the entire vector of paths
        mtm_matrix[:, j] = bsm_price(
            s_start=s_at_tj, 
            k=k, r=r, sigma=sigma, 
            t_current=t_current, 
            T_maturity=T_maturity, 
            omega=omega
        )
        
    # 4. Handle the final time step (t=T) manually as a payoff
    # Get the spot prices at maturity
    s_at_T = all_paths[:, m_steps]
    mtm_matrix[:, m_steps] = np.maximum(omega * (s_at_T - k), 0)
    
    print("MtM calculation complete.")
    return mtm_matrix, time_vector

def calculate_mtm_matrix_european_slow_analytic(
    all_paths: np.ndarray,
    k: float,
    r: float,
    sigma: float,
    T_maturity: float,
    omega: int
):
    """
    Calculates the MtM matrix for a European option given pre-simulated paths.
    This uses a "slow" double-loop approach but still calls the FAST
    analytic pricer (bsm_price) for each cell.
    """
    
    # 1. Get shape and time info
    n_sims_outer, m_steps_plus_1 = all_paths.shape
    m_steps = m_steps_plus_1 - 1
    
    # 2. Create helper variables
    mtm_matrix = np.zeros_like(all_paths)
    time_vector = np.linspace(0, T_maturity, m_steps + 1)
    
    # 3. Loop through TIME (columns)
    for j in range(m_steps):
        t_current = time_vector[j]
        
        if j % 20 == 0:
             print(f"  Pricing time step {j}/{m_steps}...")
        
        # 4. Loop through PATHS (rows)
        for i in range(n_sims_outer):
            s_current_i = all_paths[i, j]
            price_i = bsm_price(
                s_start=s_current_i,
                k=k, r=r, sigma=sigma,
                t_current=t_current,
                T_maturity=T_maturity,
                omega=omega
            )
            mtm_matrix[i, j] = price_i
            
    # 5. Handle the final time step (t=T) manually as a payoff
    s_at_T = all_paths[:, m_steps]
    mtm_matrix[:, m_steps] = np.maximum(omega * (s_at_T - k), 0)
            
    return mtm_matrix, time_vector

In [ ]:
#  Set Parameters
s_start = 100
k = 100
r = 0.05
sigma = 0.2
T_maturity = 1.0
n_sims = 10000  
m_steps = 250         
omega = 1            # 1 for a Call option
position = 1         # 1 for LONG

np.random.seed(42)



In [4]:
print("Non-vectorized Sim")
start_time = time.time()
all_paths = gbm_path_sim(s_start, r, sigma, T_maturity, n_sims, m_steps)
mtm_matrix_euro_slow, _ = calculate_mtm_matrix_european_slow_analytic(
    all_paths=all_paths, k=k, r=r, sigma=sigma,
    T_maturity=T_maturity, omega=omega
)
end_time = time.time()
print(f"Exposure simulation complete. Took {end_time - start_time:.4f} seconds.")

Non-vectorized Sim
  Pricing time step 0/250...
  Pricing time step 20/250...
  Pricing time step 40/250...
  Pricing time step 60/250...
  Pricing time step 80/250...
  Pricing time step 100/250...
  Pricing time step 120/250...
  Pricing time step 140/250...
  Pricing time step 160/250...
  Pricing time step 180/250...
  Pricing time step 200/250...
  Pricing time step 220/250...
  Pricing time step 240/250...
Exposure simulation complete. Took 125.9185 seconds.


In [6]:
print("Vectorized sim")
start_time = time.time()
all_paths = gbm_path_sim_vectorized(s_start, r, sigma, T_maturity, n_sims, m_steps)
mtm_matrix_euro_fast, time_vector = calculate_mtm_matrix_european(
    all_paths=all_paths, k=k, r=r, sigma=sigma, 
    T_maturity=T_maturity, omega=omega
)
end_time = time.time()
print(f"Exposure simulation complete. Took {end_time - start_time:.4f} seconds.")

Vectorized sim
Calculating MtM matrix (pricing at each time step)...
MtM calculation complete.
Exposure simulation complete. Took 0.2242 seconds.
